In [ ]:
# Cell 1 — Install dependencies & download dataset
!pip install open-clip-torch opencv-python-headless tqdm pyyaml pandas -q

import os, zipfile, urllib.request

os.makedirs('data/ucf101', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print('Downloading UCF-101...')
urllib.request.urlretrieve(
    'https://www.crcv.ucf.edu/data/UCF101/UCF101.rar',
    'data/ucf101/UCF101.rar'
)

print('Downloading train/test splits...')
urllib.request.urlretrieve(
    'https://www.crcv.ucf.edu/data/UCF101/UCF101TrainTestSplits-RecognitionTask.zip',
    'data/ucf101/splits.zip'
)

print('Extracting splits...')
with zipfile.ZipFile('data/ucf101/splits.zip') as z:
    z.extractall('data/ucf101/')

print('Extracting UCF-101 videos (this takes a few minutes)...')
!apt-get install -y unrar -q
!unrar x data/ucf101/UCF101.rar data/ucf101/ -y > /dev/null

print('Done!')

In [1]:
# Cell 2 — Full benchmark
import warnings
warnings.filterwarnings('ignore')

import torch
import open_clip
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
import pandas as pd
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'openai'
NUM_FRAMES = 8
MAX_PER_CLASS = 10
PROMPT_TEMPLATE = 'a video of {}'
UCF_ROOT = 'data/ucf101/UCF-101'
CLASSLIST = 'data/ucf101/ucfTrainTestlist/classInd.txt'
TESTLIST = 'data/ucf101/ucfTrainTestlist/testlist01.txt'

print(f'Device: {DEVICE}')


def load_classes(path):
    classes = {}
    with open(path) as f:
        for line in f:
            idx, name = line.strip().split()
            classes[int(idx)] = name.replace('_', ' ').lower()
    return classes


def load_test_videos(testlist, root, max_per_class):
    root = Path(root)
    videos, counts = [], {}
    with open(testlist) as f:
        for line in f:
            rel = line.strip()
            label = rel.split('/')[0].replace('_', ' ').lower()
            if counts.get(label, 0) >= max_per_class:
                continue
            full = root / rel
            if full.exists():
                videos.append({'path': str(full), 'label': label})
                counts[label] = counts.get(label, 0) + 1
    return videos


def extract_frames(path, n=8):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []
    frames = []
    for idx in np.linspace(0, total - 1, n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames


model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.to(DEVICE).eval()
print('Model loaded')

classes = load_classes(CLASSLIST)
label_list = [classes[k] for k in sorted(classes)]

texts = tokenizer([PROMPT_TEMPLATE.format(l) for l in label_list]).to(DEVICE)
with torch.no_grad():
    text_embs = model.encode_text(texts)
    text_embs = text_embs / text_embs.norm(dim=-1, keepdim=True)
print(f'Encoded {len(label_list)} class labels')

videos = load_test_videos(TESTLIST, UCF_ROOT, MAX_PER_CLASS)
print(f'Total videos to evaluate: {len(videos)}')

results = []
for video in tqdm(videos, desc='Evaluating'):
    frames = extract_frames(video['path'], NUM_FRAMES)
    if not frames:
        continue
    tensors = torch.stack([preprocess(f) for f in frames]).to(DEVICE)
    with torch.no_grad():
        embs = model.encode_image(tensors)
        embs = embs / embs.norm(dim=-1, keepdim=True)
        video_emb = embs.mean(dim=0)
    sims = (video_emb @ text_embs.T).cpu().numpy()
    top5 = [label_list[i] for i in np.argsort(sims)[::-1][:5]]
    results.append({
        'video': Path(video['path']).name,
        'true_label': video['label'],
        'top1_pred': top5[0],
        'top5_preds': top5,
        'top1_correct': video['label'] == top5[0],
        'top5_correct': video['label'] in top5,
    })

df = pd.DataFrame(results)
df.to_csv('outputs/results.csv', index=False)

top1 = df['top1_correct'].mean() * 100
top5 = df['top5_correct'].mean() * 100
per_class = df.groupby('true_label')['top1_correct'].mean() * 100

metrics = {
    'model': f'{MODEL_NAME} ({PRETRAINED})',
    'num_frames': NUM_FRAMES,
    'top1_accuracy': round(top1, 2),
    'top5_accuracy': round(top5, 2),
    'total_videos': len(df),
    'best_classes': per_class.nlargest(10).round(1).to_dict(),
    'worst_classes': per_class.nsmallest(10).round(1).to_dict(),
}
with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nTop-1 Accuracy : {top1:.2f}%')
print(f'Top-5 Accuracy : {top5:.2f}%')
print(f'Total Videos   : {len(df)}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Zero-Shot CLIP Benchmark — UCF-101 ({MODEL_NAME})', fontsize=13, fontweight='bold')

axes[0].bar(['Top-1', 'Top-5'], [top1, top5], color=['#4C72B0', '#55A868'], width=0.4)
axes[0].set_ylim(0, 100)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title('Overall Accuracy')
for i, v in enumerate([top1, top5]):
    axes[0].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

best10 = per_class.nlargest(10).sort_values()
axes[1].barh(best10.index, best10.values, color='#55A868')
axes[1].set_xlim(0, 105)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Top 10 Best Classes')

worst10 = per_class.nsmallest(10).sort_values(ascending=False)
axes[2].barh(worst10.index, worst10.values, color='#C44E52')
axes[2].set_xlim(0, 105)
axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[2].set_title('Top 10 Worst Classes')

plt.tight_layout()
plt.savefig('outputs/benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to outputs/benchmark_results.png')

Device: cpu
Model loaded


FileNotFoundError: [Errno 2] No such file or directory: 'data/ucf101/ucfTrainTestlist/classInd.txt'